# Laboratorio 12b — Una mini-GAN
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 25 de noviembre de 2026 · Clase 12 · Lab. Central 114
**Bloque:** Lab B · ~12:20–13:00 (40 min), después de la teoría GAN
**Viene de:** teoría GAN (deck «Clase 12 - RNN y GAN», diapos 13–18: falsificador contra policía, entrenamiento por turnos y colapso de modos).
**Antes:** Lab 12a (pronóstico con RNN, LSTM y GRU, otro notebook).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **B1** | GAN en 1D: los dos montes | 20 min |
| **B2** | Dígitos inventados (opcional) | 15 min |
| **B3** | Policía demasiado rápido y reflexión | 5 min |
| **Entregable** | Tabla de tres escenarios | al final |

## Objetivos de aprendizaje

1. Entrenar por turnos un falsificador (generador) y un policía (discriminador).
2. Ver con datos propios lo de la diapo 16: una GAN sana aprende **los dos montes**.
3. Provocar un **colapso de modos** (el falsificador se queda con un solo monte) haciendo que el policía aprenda demasiado rápido, y aprender a detectarlo mirando lo generado, no solo las curvas.

> Todo el notebook es **offline**: los números se inventan aquí y los dígitos vienen incluidos en scikit-learn. Funciona en Colab o en tu computador (basta la CPU).

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy pandas matplotlib scikit-learn torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch
from IPython.display import HTML, display
from sklearn.datasets import load_digits
try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("No se encontró PyTorch. En tu computador: activa el entorno virtual e instala con "
                      "`pip install -r requirements.txt`. Si no, abre este notebook en Google Colab.") from e

SEED = 42
RAPIDO = False   # True = menos pasos, solo para ensayar el código (los valores del libreto usan False)
torch.set_num_threads(1)
T0 = time.time()


def fijar_semillas(semilla=SEED):
    """Deja reproducibles random, NumPy y PyTorch."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)


fijar_semillas()

# Paleta del deck «Clase 12 - RNN y GAN»
AZUL, NARANJA, MARINO, GRIS, GRIS_CLARO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#d9dbdf"
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__, "| RAPIDO =", RAPIDO, "| Colab =", IN_COLAB)

---
## El juego: falsificador contra policía

> **En palabras simples (diapos 14 y 15).** Dos redes juegan por turnos. El generador (falsificador) recibe ruido al azar y fabrica números falsos. El discriminador (policía) mira un número y dice qué tan seguro está de que es real. En su turno, el policía practica distinguir; en el suyo, el falsificador ajusta su receta para engañarlo. Si todo sale bien, al final el policía acierta la mitad: ya no puede distinguir.

Es el mismo dibujo de la diapo 14:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.set(xlim=(0, 10), ylim=(-0.45, 3.0))
ax.axis("off")


def caja(x, y, titulo, detalle, fondo, texto="white", borde="none"):
    ax.add_patch(FancyBboxPatch((x, y), 1.9, 0.9, boxstyle="round,pad=0.04,rounding_size=0.15", fc=fondo, ec=borde, lw=1.8))
    ax.text(x + 0.95, y + 0.58, titulo, ha="center", va="center", color=texto, fontsize=11.5, fontweight="bold")
    ax.text(x + 0.95, y + 0.27, detalle, ha="center", va="center", color=texto, fontsize=9)


def flecha(x0, y0, x1, y1, **kw):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color=MARINO, lw=1.8, **kw))


caja(0.1, 1.3, "ruido", "8 números al azar", GRIS)
caja(2.6, 1.3, "G · falsificador", "fabrica 1 número", NARANJA)
caja(5.1, 1.3, "número falso", "lo que fabricó G", GRIS_CLARO, texto=MARINO, borde=NARANJA)
caja(5.1, -0.25, "número real", "de los dos montes", GRIS_CLARO, texto=MARINO, borde=MARINO)
caja(7.8, 0.55, "D · policía", "¿qué tan real? (0 a 1)", AZUL)
flecha(2.0, 1.75, 2.6, 1.75)
flecha(4.5, 1.75, 5.1, 1.75)
flecha(7.0, 1.75, 7.8, 1.25)
flecha(7.0, 0.2, 7.8, 0.85)
ax.annotate("", xy=(3.55, 2.2), xytext=(8.75, 1.45),
            arrowprops=dict(arrowstyle="-|>", color=NARANJA, lw=1.8, ls="--", connectionstyle="arc3,rad=0.25"))
ax.text(6.4, 2.75, "pistas: «así me engañarías mejor»", ha="center", va="center", color=NARANJA, fontsize=10, style="italic")
plt.show()

---
## Parte B1 — GAN en 1D: los dos montes (20 min)

> **En palabras simples (diapo 16).** Los datos reales son 10.000 números que forman **dos montes (dos campanas)**: uno alrededor de −2 y otro alrededor de +2, cada uno de ancho 0,5. El falsificador nunca ve esos números: solo recibe las pistas del policía. Si el juego sale bien, termina fabricando números en los dos montes. Si sale mal, aparece el **colapso de modos**: el falsificador se queda con un solo monte.

**Consigna**

1. Mira primero los datos reales: es lo que el falsificador tiene que aprender a imitar.
2. Arma las dos redes. El ruido son **8 números al azar**; el generador los transforma en **1 número**. El policía recibe 1 número y entrega una probabilidad (de 0 a 1) de que sea real.
3. Completa los **turnos** del entrenamiento y entrena **3000 pasos** (lotes de 128, Adam con `lr=1e-3` para las dos redes).
4. Mira cómo aparecen los montes (animación y grilla cada 500 pasos) y mide qué fracción de lo generado cae en cada monte.

In [ ]:
def muestrear_real(n):
    """Dos montes del mismo tamaño: uno en −2 y otro en +2, cada uno una campana de ancho 0,5. Devuelve (n, 1)."""
    elige_izq = np.random.rand(n) < 0.5
    x = np.where(elige_izq, np.random.normal(-2, 0.5, n), np.random.normal(2, 0.5, n))
    return torch.tensor(x, dtype=torch.float32).unsqueeze(1)


fijar_semillas(SEED)
datos_reales = muestrear_real(10_000)
DIM_Z = 8                                   # el ruido: 8 números al azar
xs = np.linspace(-5, 5, 400)


def densidad_real(x):
    """La forma exacta de los dos montes (para dibujarla debajo de los histogramas)."""
    campana = lambda centro: np.exp(-0.5 * ((x - centro) / 0.5) ** 2) / (0.5 * np.sqrt(2 * np.pi))
    return 0.5 * campana(-2) + 0.5 * campana(2)


def cobertura_modos(muestras):
    """Fracción de números a menos de 1 del monte −2, del monte +2 y fuera de ambos."""
    m = np.asarray(muestras).ravel()
    izq, der = np.mean(np.abs(m + 2) < 1), np.mean(np.abs(m - 2) < 1)
    return {"monte −2": izq, "monte +2": der, "fuera de ambos": 1 - izq - der}


def dibujar_real(ax):
    """Datos reales: gris claro relleno con borde azul marino (como la diapo 16)."""
    ax.fill_between(xs, densidad_real(xs), color=GRIS_CLARO, label="real")
    ax.plot(xs, densidad_real(xs), color=MARINO, lw=1.6)


def dibujar_generado(ax, muestras):
    """Lo generado: histograma naranja."""
    ax.hist(np.asarray(muestras).ravel(), bins=60, range=(-5, 5), density=True, color=NARANJA, alpha=0.75, label="generado")


fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(datos_reales.numpy().ravel(), bins=60, range=(-5, 5), density=True, histtype="stepfilled",
        fc=GRIS_CLARO, ec=MARINO, lw=1.4)
ax.set(title="Esto es lo que el falsificador tiene que aprender a imitar", xlabel="valor", ylabel="densidad")
plt.show()
print("Datos reales:", tuple(datos_reales.shape), "| cobertura:",
      {k: round(float(v), 2) for k, v in cobertura_modos(datos_reales.numpy()).items()})

In [ ]:
def crear_generador(dim_z=DIM_Z):
    """Falsificador: 8 números al azar -> 1 número."""
    # TODO: devuelve nn.Sequential(Linear(dim_z, 32) -> ReLU -> Linear(32, 1))
    ...


def crear_discriminador():
    """Policía: 1 número -> probabilidad (0 a 1) de que sea real."""
    # TODO: devuelve nn.Sequential(Linear(1, 32) -> LeakyReLU(0.2) -> Linear(32, 1) -> Sigmoid)
    ...

In [ ]:
# --- Autochequeo: las dos redes ---
g, d = crear_generador(), crear_discriminador()
assert g is not None and d is not None, "Falta completar crear_generador o crear_discriminador (¿olvidaste el return?)."
assert g(torch.randn(5, DIM_Z)).shape == (5, 1), "El falsificador debería convertir 5 ruidos de 8 números en 5 números: forma (5, 1)."
salida_d = d(torch.randn(5, 1))
assert salida_d.shape == (5, 1), "El policía debería entregar un número por cada número que mira: forma (5, 1)."
assert 0 <= salida_d.min().item() and salida_d.max().item() <= 1, "El policía debe entregar probabilidades entre 0 y 1: ¿falta la Sigmoid final?"
assert sum(p.numel() for p in g.parameters()) == 8 * 32 + 32 + 32 + 1, "El falsificador no tiene el tamaño esperado (8 -> 32 -> 1)."
assert sum(p.numel() for p in d.parameters()) == 32 + 32 + 32 + 1, "El policía no tiene el tamaño esperado (1 -> 32 -> 1)."
print("✔ Falsificador y policía bien")

### Los turnos

En cada paso del entrenamiento pasan dos cosas, en este orden:

1. **Turno del policía.** Mira un lote de números reales (debería decir 1) y uno de falsos (debería decir 0), y se corrige.
2. **Turno del falsificador.** Fabrica números y se corrige para que el policía diga 1 («real») sobre ellos.

`bce` mide qué tan lejos está la respuesta del policía de lo que debería haber dicho.

In [ ]:
PASOS_GAN = 1000 if RAPIDO else 3000


def entrenar_gan(pasos=PASOS_GAN, lr_g=1e-3, lr_d=1e-3, lote=128, foto_cada=100, semilla=SEED):
    """Entrena por turnos. Devuelve G, D, las pérdidas de cada paso y una «foto» de 5000 números generados cada `foto_cada` pasos."""
    fijar_semillas(semilla)
    G, D = crear_generador(), crear_discriminador()
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d)
    bce = nn.BCELoss()
    unos, ceros = torch.ones(lote, 1), torch.zeros(lote, 1)
    hist, fotos = {"D": [], "G": []}, {}
    for paso in range(1, pasos + 1):
        real = datos_reales[torch.randint(0, len(datos_reales), (lote,))]
        falso = G(torch.randn(lote, DIM_Z))

        # Turno del policía: aprende a decir 1 a lo real y 0 a lo falso.
        # (falso.detach(): en este turno solo se corrige al policía, no al falsificador)
        opt_d.zero_grad()
        # TODO: suma dos errores: bce(lo que dice D sobre `real`, unos) + bce(lo que dice D sobre falso.detach(), ceros)
        loss_d = ...
        loss_d.backward()
        opt_d.step()

        # Turno del falsificador: ajusta su receta para que el policía diga 1 («real») sobre lo falso.
        opt_g.zero_grad()
        # TODO: un solo error: bce(lo que dice D sobre `falso`, unos)   (aquí SIN .detach(): ahora se corrige al falsificador)
        loss_g = ...
        loss_g.backward()
        opt_g.step()

        hist["D"].append(loss_d.item())
        hist["G"].append(loss_g.item())
        if paso % foto_cada == 0:
            with torch.no_grad():
                fotos[paso] = G(torch.randn(5000, DIM_Z)).numpy().ravel()
    return G, D, hist, fotos

In [ ]:
# --- Autochequeo: los turnos (un ensayo de 20 pasos) ---
try:
    _G, _D, _h, _f = entrenar_gan(pasos=20, foto_cada=10)
except AttributeError as e:
    raise AssertionError("Falta completar una de las dos líneas de los turnos (todavía hay `...`).") from e
assert len(_h["D"]) == len(_h["G"]) == 20 and sorted(_f) == [10, 20], "Debería quedar una pérdida por paso y una foto cada 10 pasos."
assert abs(_h["D"][0] - 2 * np.log(2)) < 0.35, \
    f"Al comienzo la pérdida del policía debería rondar 1,39 (suma de DOS errores) y es {_h['D'][0]:.2f}. ¿Sumaste lo real y lo falso?"
assert abs(_h["G"][0] - np.log(2)) < 0.35, \
    f"Al comienzo la pérdida del falsificador debería rondar 0,69 y es {_h['G'][0]:.2f}. ¿Comparaste D(falso) con `unos`?"
print("✔ Los turnos están bien armados")

### Entrenar (unos segundos)

In [ ]:
t0 = time.time()
G1, D1, hist1, fotos1 = entrenar_gan()
print(f"Entrenamiento listo en {time.time() - t0:.0f} s")

### La GAN aprendiendo, en movimiento

Es la diapo 16 (panel «sano») con los datos de este lab: el histograma **naranja** es lo que fabrica el falsificador; la curva **gris con borde azul marino**, los datos reales. Un cuadro cada 100 pasos.

In [ ]:
pasos_foto = sorted(fotos1)
fig, ax = plt.subplots(figsize=(7.5, 3.3), dpi=80)


def cuadro(k):
    ax.cla()
    dibujar_real(ax)
    dibujar_generado(ax, fotos1[pasos_foto[k]])
    ax.set(xlim=(-5, 5), ylim=(0, 1.0), xlabel="valor", ylabel="densidad",
           title=f"paso {pasos_foto[k]} de {PASOS_GAN}")
    ax.legend(loc="upper right", frameon=False)


fig.subplots_adjust(bottom=0.17)
anim = animation.FuncAnimation(fig, cuadro, frames=len(pasos_foto), interval=300)
plt.close(fig)                            # evita que la figura salga repetida debajo de la animación
HTML(anim.to_jshtml())

### Para el entregable: la grilla cada 500 pasos y la cobertura de los montes

In [ ]:
pasos_500 = [p for p in pasos_foto if p % 500 == 0]
fig, ejes = plt.subplots(2, 3, figsize=(11, 5.2), sharex=True, sharey=True)
for ax in ejes.ravel()[len(pasos_500):]:
    ax.axis("off")
for ax, paso in zip(ejes.ravel(), pasos_500):
    dibujar_real(ax)
    dibujar_generado(ax, fotos1[paso])
    ax.set(xlim=(-5, 5), ylim=(0, 1.0), title=f"paso {paso}")
ejes[0, 0].legend(frameon=False, fontsize=9)
fig.supxlabel("valor")
fig.supylabel("densidad")
plt.tight_layout()
plt.show()

cob = pd.DataFrame({f"paso {p}": cobertura_modos(fotos1[p]) for p in pasos_500}).T
cob.loc["datos reales"] = cobertura_modos(datos_reales.numpy())
display(cob.round(2))

### Las curvas de pérdida

Aquí las curvas **no tienen que bajar**: las dos redes se persiguen. Las líneas punteadas marcan el empate ideal (policía ≈ 1,39, falsificador ≈ 0,69). Por qué justo esos números está en «Para saber más», al final.

In [ ]:
def suavizar(v, k=50):
    """Promedio móvil de k pasos, para que las curvas se lean mejor."""
    return np.convolve(v, np.ones(k) / k, mode="valid")


fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.plot(suavizar(hist1["D"]), color=AZUL, label="policía (D)")
ax.plot(suavizar(hist1["G"]), color=NARANJA, label="falsificador (G)")
ax.axhline(2 * np.log(2), color=AZUL, ls=":", lw=1.2)
ax.axhline(np.log(2), color=NARANJA, ls=":", lw=1.2)
ax.set(title="Pérdidas durante el juego (promedio de 50 pasos)", xlabel="paso", ylabel="pérdida (BCE)")
ax.legend(frameon=False)
plt.show()
print(f"Promedio de los últimos 500 pasos -> policía: {np.mean(hist1['D'][-500:]):.2f} (empate 1,39) "
      f"| falsificador: {np.mean(hist1['G'][-500:]):.2f} (empate 0,69)")

In [ ]:
# --- Autochequeo B1 ---
assert len(hist1["D"]) == len(hist1["G"]) == PASOS_GAN, "Debería haber una pérdida por paso."
assert len(fotos1) == PASOS_GAN // 100, "Debería haber una foto cada 100 pasos."
final = cobertura_modos(fotos1[pasos_foto[-1]])
if RAPIDO:
    print("⚠ RAPIDO = True: me salto el chequeo de los dos montes (con 1000 pasos la GAN todavía no termina de aprender).")
else:
    assert final["monte −2"] >= 0.35 and final["monte +2"] >= 0.35 and final["fuera de ambos"] <= 0.20, (
        f"El falsificador todavía no aprende los dos montes ({final['monte −2']:.2f} / {final['monte +2']:.2f} / "
        f"{final['fuera de ambos']:.2f}). Revisa la tasa de aprendizaje (1e-3) y el orden de los turnos.")
    print(f"✔ B1 OK: monte −2 = {final['monte −2']:.2f} | monte +2 = {final['monte +2']:.2f} | fuera = {final['fuera de ambos']:.2f}")

**Pregunta B1.** ¿Desde qué paso se ven claramente los dos montes? Al final, ¿qué fracción de lo generado quedó fuera de ambos, comparado con los datos reales?

**Respuesta:** (escribe aquí)

---
## Parte B2 — Dígitos inventados (15 min, opcional)

> **En palabras simples (diapo 14).** Lo mismo, pero ahora el falsificador dibuja dígitos de 8×8 píxeles. Cada fila de la grilla es una etapa del entrenamiento: miren cómo pasan de manchas a números.

**La receta:** el falsificador recibe 16 números al azar y entrega 64 píxeles (16 → 64 → 64, con ReLU en el medio y Sigmoid al final para que cada píxel quede entre 0 y 1). El policía mira los 64 píxeles (64 → 32 → 1, con LeakyReLU(0,2) y Sigmoid). Son 5000 pasos con lotes de 64. Las fotos usan siempre el **mismo ruido**, para ver cómo cambia un mismo dibujo.

In [ ]:
PASOS_DIG = 1500 if RAPIDO else 5000
digitos = torch.tensor(load_digits().data / 16.0, dtype=torch.float32)     # (1797, 64), píxeles entre 0 y 1


def entrenar_gan_digitos(pasos=PASOS_DIG, lote=64, lr=2e-4, foto_cada=1000, dim_z=16, semilla=SEED):
    fijar_semillas(semilla)
    # TODO: define G (dim_z -> 64 -> 64, ReLU en el medio y Sigmoid al final)
    #       y D (64 -> 32 -> 1, LeakyReLU(0.2) en el medio y Sigmoid al final), con nn.Sequential.
    G, D = ..., ...
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCELoss()
    unos, ceros = torch.ones(lote, 1), torch.zeros(lote, 1)
    z_fijo = torch.randn(10, dim_z)            # mismo ruido en todas las fotos
    fotos = {}
    for paso in range(1, pasos + 1):
        real = digitos[torch.randint(0, len(digitos), (lote,))]
        falso = G(torch.randn(lote, dim_z))
        # TODO: los mismos dos turnos de B1 (primero el policía, con falso.detach(); después el falsificador).
        #       Cada turno: zero_grad, calcular la pérdida, backward y step.
        ...
        if paso % foto_cada == 0:
            with torch.no_grad():
                fotos[paso] = G(z_fijo).numpy().reshape(-1, 8, 8)
    return G, D, fotos

In [ ]:
# --- Autochequeo B2 (ensayo de 30 pasos) ---
try:
    _G0, _, _ = entrenar_gan_digitos(pasos=0)
    _G1, _D1, _ = entrenar_gan_digitos(pasos=30)
except AttributeError as e:
    raise AssertionError("Falta completar G y D en `entrenar_gan_digitos` (todavía hay `...`).") from e
_z = torch.randn(4, 16)
assert _G1(_z).shape == (4, 64) and _D1(_G1(_z)).shape == (4, 1), "G debería entregar 64 píxeles y D una probabilidad por imagen."
assert not torch.allclose(_G0(_z), _G1(_z)), "El falsificador no cambió en 30 pasos: falta el ciclo de turnos."
print("✔ GAN de dígitos bien armada")

In [ ]:
t0 = time.time()
G3, D3, fotos3 = entrenar_gan_digitos()
print(f"Entrenamiento listo en {time.time() - t0:.0f} s")

filas_img = [("reales", load_digits().images[:10] / 16.0)] + [(f"paso {p}", imgs) for p, imgs in sorted(fotos3.items())]
fig, ejes = plt.subplots(len(filas_img), 10, figsize=(9, 0.95 * len(filas_img) + 0.6))
for r, (nombre, imgs) in enumerate(filas_img):
    for c in range(10):
        ejes[r, c].imshow(imgs[c], cmap="gray_r", vmin=0, vmax=1)
        ejes[r, c].set_xticks([]); ejes[r, c].set_yticks([]); ejes[r, c].grid(False)
    ejes[r, 0].set_ylabel(nombre, rotation=0, ha="right", va="center", fontsize=10)
plt.suptitle("Arriba, dígitos reales. Abajo, lo que dibuja el falsificador con el mismo ruido a lo largo del entrenamiento", fontsize=10)
plt.tight_layout()
plt.show()

**Pregunta B2.** ¿En qué fila empiezan a parecer dígitos? ¿Los 10 dibujos de la última fila son variados o se repiten?

**Respuesta:** (escribe aquí)

---
## Parte B3 — Policía demasiado rápido y reflexión (5 min)

> **En palabras simples (diapo 16).** ¿Qué pasa si el policía aprende 10 veces más rápido que el falsificador? Gana demasiado fácil y sus pistas dejan de servir. Al falsificador le conviene esconderse en un solo monte donde todavía engaña: es el **colapso de modos**, el falsificador se queda con un solo monte.

**Experimento:** misma receta de B1, pero el policía con `lr_d=1e-2` (10 veces más rápido) y el falsificador igual, `lr_g=1e-3`.

In [ ]:
G2, D2, hist2, fotos2 = entrenar_gan(lr_d=1e-2, lr_g=1e-3)
_, _, _, fotos2_s1 = entrenar_gan(lr_d=1e-2, lr_g=1e-3, semilla=1)     # el mismo experimento, con otra semilla

ultimo = pasos_foto[-1]
fig, ejes = plt.subplots(2, 2, figsize=(11.5, 6), gridspec_kw={"height_ratios": [1.3, 1]})
for col, (titulo, fotos, hist) in enumerate([("Receta del lab (los dos a 1e-3): sano", fotos1, hist1),
                                              ("Policía 10 veces más rápido (1e-2): colapso", fotos2, hist2)]):
    ax = ejes[0, col]
    dibujar_real(ax)
    dibujar_generado(ax, fotos[ultimo])
    ax.set(xlim=(-5, 5), title=f"{titulo}\npaso {ultimo}", xlabel="valor", ylabel="densidad")
    ax.legend(loc="upper left", frameon=False, fontsize=9)
    ax = ejes[1, col]
    ax.plot(suavizar(hist["D"]), color=AZUL, label="policía (D)")
    ax.plot(suavizar(hist["G"]), color=NARANJA, label="falsificador (G)")
    ax.axhline(2 * np.log(2), color=AZUL, ls=":", lw=1.2)
    ax.axhline(np.log(2), color=NARANJA, ls=":", lw=1.2)
    ax.set(title="pérdidas (promedio de 50 pasos)", xlabel="paso", ylabel="BCE", ylim=(0.4, 2.2))
    ax.legend(loc="upper right", frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

comparacion = pd.DataFrame({
    "receta del lab (lr_d = 1e-3)": cobertura_modos(fotos1[ultimo]),
    "policía rápido (lr_d = 1e-2)": cobertura_modos(fotos2[ultimo]),
    "policía rápido, semilla 1": cobertura_modos(fotos2_s1[ultimo]),
    "datos reales": cobertura_modos(datos_reales.numpy()),
}).T
display(comparacion.round(2))
for nombre, h in [("receta del lab", hist1), ("policía rápido", hist2)]:
    print(f"{nombre:>15}, últimos 500 pasos -> policía: {np.mean(h['D'][-500:]):.2f} | falsificador: {np.mean(h['G'][-500:]):.2f}")

In [ ]:
# --- Autochequeo B3 ---
c2 = cobertura_modos(fotos2[ultimo])
if RAPIDO:
    print("⚠ RAPIDO = True: me salto el chequeo del colapso (con 1000 pasos todavía puede no verse).")
else:
    assert max(c2["monte −2"], c2["monte +2"]) >= 0.9, \
        "Con el policía 10 veces más rápido el falsificador debería quedarse con un solo monte (≥ 0,9 en uno). Revisa lr_d=1e-2."
    print(f"✔ B3 OK: colapso a un monte ({c2['monte −2']:.2f} / {c2['monte +2']:.2f})")

**Pregunta B3.1.** ¿Qué pasó cuando el policía aprendía 10 veces más rápido? ¿Cuántos montes aprendió el falsificador?

**Respuesta:** (escribe aquí)

**Pregunta B3.2.** Si solo miraras las curvas de pérdida, ¿te darías cuenta del problema? ¿Qué gráfico sí lo muestra?

**Respuesta:** (escribe aquí)

**Pregunta B3.3.** ¿En qué se parece y en qué se diferencia una GAN de un modelo de lenguaje como ChatGPT?

**Respuesta:** (escribe aquí)

---
# Entregable

Sube a **AULA-USM** los **dos notebooks estudiante ejecutados**:

1. `Lab12a_Pronostico_estudiante.ipynb`: A1–A4 con respuestas.
2. `Lab12b_MiniGAN_estudiante.ipynb` (este): B1 y B3 con respuestas (B2 es opcional).
3. La tabla de abajo, completa.

**¿Qué red uso?** (el árbol de la diapo 19)

> ¿tabla? → **MLP** · ¿imágenes? → **CNN** · ¿datos en orden? → **RNN/LSTM/GRU** · ¿crear datos nuevos? → **GAN**

Completa la columna «Arquitectura» con **MLP, CNN, RNN/LSTM/GRU o GAN** y justifica en una línea:

| Escenario | Arquitectura | Justificación |
|---|---|---|
| **a)** Clasificar la cobertura de suelo (bosque, cultivo, urbano, agua) de imágenes satelitales | | |
| **b)** Pronosticar la demanda eléctrica horaria de una comuna para las próximas 24 horas | | |
| **c)** Generar datos sintéticos de sensores para entrenar un detector de fallas muy poco frecuentes | | |

In [ ]:
print(f"Tiempo total del notebook: {time.time() - T0:.0f} s")

---
## Para saber más (opcional): la fórmula del juego, en palabras

Nada de esto es necesario para el lab. Es lo mismo del Anexo B del deck (diapo 21).

$$\min_G\ \max_D\ \ \mathbb{E}_{x\,\sim\,\text{datos}}\big[\log D(x)\big] \;+\; \mathbb{E}_{z\,\sim\,\text{ruido}}\big[\log\big(1 - D(G(z))\big)\big]$$

| Pieza | En palabras |
|---|---|
| $D(x)$ | Lo que dice el policía sobre un número real $x$: de 0 a 1, qué tan seguro está de que es real. |
| $G(z)$ | El número falso que fabrica el falsificador a partir del ruido $z$ (los 8 números al azar). |
| $D(G(z))$ | Lo que dice el policía sobre un número falso. |
| $\mathbb{E}[\ \cdot\ ]$ | «En promedio», sobre muchos ejemplos (en el código: el promedio de un lote). |
| $\log D(x)$ | Premio del policía por reconocer lo real: vale 0 si dice 1, y se hunde hacia −∞ si dice 0. |
| $\log(1 - D(G(z)))$ | Premio del policía por descubrir lo falso: vale 0 si dice 0 ante un falso. |
| $\max_D$ | El policía quiere que la suma sea **lo más alta posible** (acertar en los dos casos). |
| $\min_G$ | El falsificador quiere lo contrario: que sea **lo más baja posible** (que el policía se equivoque con lo falso). |

En el código no se escribe así: `bce` es ese mismo premio con el signo cambiado (una pérdida, que se baja). Además, en su turno el falsificador usa el truco de pedirle al policía que diga 1 sobre lo falso (`bce(D(falso), unos)`), que da pistas más fuertes al comienzo, cuando el policía todavía lo pilla con facilidad.

**¿Por qué el empate ideal es 1,39 y 0,69?** En el empate, el policía ya no distingue y responde 0,5 a todo. La pérdida `bce` de una respuesta de 0,5 vale $-\ln 0{,}5 = \ln 2 \approx 0{,}69$, sea el número real o falso. El falsificador tiene un solo error ($\approx 0{,}69$); el policía suma dos, uno por lo real y otro por lo falso ($2 \ln 2 \approx 1{,}39$).